# Week 10 · Day 5 — Why Sequences? The Limits of Images

For three weeks, every input has been a **fixed grid of pixels**. Today we meet a different kind of data — where **order carries the meaning** — and see why our current tools don't fit. This is the pivot from *vision* to *language & time series*, and the on-ramp to the rest of the course.

**Today:**
1. Sequential data is everywhere
2. Why CNNs/MLPs don't fit sequences
3. The sequence-model taxonomy (the four shapes)
4. The core problem: turning words into numbers
5. Where we're headed (RNNs → embeddings → attention → Transformers)

> This is a **concepts day** — mostly reading and small demos. Follow along, run the cells, and do the solo task at the end.

---
## 1. Sequential data is everywhere

A **sequence** is data where the **order matters**. Some kinds you already know:

- **Text** — word order changes meaning: *"dog bites man"* vs *"man bites dog"* — same words, opposite meaning.
- **Speech / audio** — a waveform over time; the order of sounds makes words.
- **Time series** — stock prices, weather, sensor readings: today depends on yesterday.
- **Music** — notes in sequence; shuffle them and the melody is gone.
- **Video** — a sequence of image frames (this is where vision meets sequences).

The common thread: **you can't shuffle the pieces without destroying the meaning.** That's the opposite of a single image, where the whole thing arrives at once.

<img src="images/sequence_data.jpg">

In [ ]:
# tiny demo: order changes meaning
s1 = "dog bites man"
s2 = "man bites dog"
print("same words?", sorted(s1.split()) == sorted(s2.split()))
print("same meaning?", "NO — order flips who does the biting")

# a sequence has a length that varies
for sentence in ["hi", "how are you", "this sentence is a bit longer"]:
    print(f"'{sentence}' -> {len(sentence.split())} words")

Notice two things that break our old tools:
- **Order matters** (shuffling changes meaning).
- **Length varies** (2 words, or 20) — but a CNN/MLP needs a **fixed-size** input.

---
## 2. Why CNNs and MLPs don't fit sequences

Our vision models have two assumptions that **break** on sequences:

- **They expect a fixed-size input.** A 64×64 image is always 64×64. But sentences are 3 words or 30 — there's no fixed size.
- **They treat positions independently-ish.** A dense layer has a separate weight for each input position; it has no built-in notion of "this came *before* that," and no memory carried along the sequence.

**What a sequence needs instead:**
- Read the input **one step at a time** (word by word, or timestep by timestep).
- Carry a **memory** (a "hidden state") that summarizes everything seen so far.
- Handle **any length** with the **same** small set of weights, reused at every step.

That description — *read one step, keep a memory, reuse the weights* — is exactly a **Recurrent Neural Network (RNN)**, which we start next week.

<img src="images/rnn.jpg">

<img src="images/ann_vs_rnn_vs_cnn.jpg">

<img src="images/ann_vs_rnn_vs_cnn2.jpg">

---
## 3. The sequence-model taxonomy: four shapes

Sequence problems come in **four shapes**, depending on whether the input and/or output is a sequence. Copy this table — you'll use it constantly.

| Shape | Input → Output | Real example |
|---|---|---|
| **One-to-one** | single → single | ordinary image classification (not really a sequence) |
| **One-to-many** | single → sequence | **image captioning** (one image → a sentence) |
| **Many-to-one** | sequence → single | **sentiment analysis** (a review → positive/negative) |
| **Many-to-many** | sequence → sequence | **translation** (English sentence → French sentence), **named-entity recognition** |

**Tip:** the shape tells you how to build the model. "Many-to-one" reads a whole sequence then outputs once; "many-to-many" outputs at (almost) every step.

<img src="images/one_to_one.jpg">

<img src="images/types_of_neural_networks.jpg">

In [1]:
# match each task to its shape (run and check your intuition)
tasks = {
    "Image captioning":       "one-to-many",
    "Sentiment analysis":     "many-to-one",
    "Machine translation":    "many-to-many",
    "Named-entity recognition": "many-to-many",
    "Music generation":       "one-to-many",
}
for task, shape in tasks.items():
    print(f"{task:28s} -> {shape}")

Image captioning             -> one-to-many
Sentiment analysis           -> many-to-one
Machine translation          -> many-to-many
Named-entity recognition     -> many-to-many
Music generation             -> one-to-many


---
## 4. The core problem: turning words into numbers

Before any model can read text, **words must become numbers.** How we do that turns out to matter enormously. Two naive attempts — and why they fail:

### Attempt 1: integer IDs
Give each word a number: `cat = 5`, `dog = 6`, `apple = 7`…
- **Problem:** this invents a fake ordering and fake distances. Is `cat (5)` really "less than" `dog (6)`? Is `apple` twice `cat`? The numbers imply relationships that don't exist.

### Attempt 2: one-hot vectors
Give each word a vector that's all 0s except a single 1 at its index.
- `cat = [0,0,0,0,1,0,0,…]`, `dog = [0,0,0,0,0,1,0,…]`
- **Problems:**
  - **Huge & sparse** — a 50,000-word vocabulary means 50,000-long vectors, almost all zeros.
  - **No notion of similarity** — every word is exactly the same distance from every other. "king" is no closer to "queen" than it is to "banana." The representation knows nothing about meaning.

In [ ]:
import numpy as np

# one-hot: every word equally distant from every other
vocab = ["king", "queen", "banana"]
onehot = np.eye(len(vocab))
print("one-hot vectors:\n", onehot)

def distance(a, b):
    return np.linalg.norm(a - b)

print("\ndistance king<->queen: ", distance(onehot[0], onehot[1]))
print("distance king<->banana:", distance(onehot[0], onehot[2]))
print("\n-> identical distances. one-hot has NO idea 'king' and 'queen' are related.")

### What we actually need: **embeddings**

We want a representation where:
- Each word is a **dense vector** (say 100 numbers, not 50,000).
- **Similar words are close together** — "king" near "queen," "banana" far away.
- **Meaning becomes geometry** — the famous result: **king − man + woman ≈ queen.**

That's a **word embedding**, and it's the first thing we build next week. It's what makes modern NLP possible.

<img src="images/word_embeddings.png">

---
## 5. Where we're headed

The next stretch of the course (following **Andrew Ng's Sequence Models**) builds up the modern NLP stack, in order:

1. **RNNs, GRUs, LSTMs** — models that read sequences and carry memory. *(the "read one step, keep a memory" idea)*
2. **Word embeddings** — Word2Vec, GloVe; turning words into meaningful vectors.
3. **Attention & sequence-to-sequence** — letting a model focus on the relevant parts of a long input (translation, etc.).
4. **Transformers** — the architecture behind every modern LLM, plus Hugging Face in practice.

### The link back to this week: *reuse, don't reinvent*
- In vision, we reused a pretrained CNN (transfer learning).
- In language, we'll reuse pretrained **embeddings** and pretrained **Transformers** — the **same mindset, new domain.**
- Every big idea you learned for images has a language counterpart coming up.

<img src="images/rnn_to_transformers.png">

<img src="images/where_we_are_hidding.png">

---
## Your turn (solo task) ✍️

**Part A — classify the shape.** For each task, name the sequence-model shape (*one-to-one / one-to-many / many-to-one / many-to-many*) and write one sentence justifying it:
1. **Sentiment analysis** of a movie review.
2. **Machine translation** (English → Urdu).
3. **Image captioning** (photo → sentence).

**Part B — think it through.** In 2–3 sentences: *why can't we just feed a sentence into the CNN we built last week?* (Hint: think about fixed size and order.)

**Part C — optional.** Write down one example of sequential data from your own life or field that we didn't mention, and say which shape a useful task on it would be.

Write your answers in the cell below.

**Your answers:**

*Part A:*
1. 
2. 
3. 

*Part B:*


*Part C:*


---
## Summary

- **Sequential data** (text, speech, time series, music, video) has meaning in its **order** — you can't shuffle it.
- **CNNs/MLPs don't fit:** they need a **fixed-size** input and have **no memory** of order. Sequences need a model that **reads one step at a time and carries a memory** — an **RNN**.
- **Four shapes:** one-to-one, one-to-many (captioning), many-to-one (sentiment), many-to-many (translation).
- **Words → numbers is the core NLP problem.** Integer IDs and one-hot vectors fail (fake ordering; no similarity). We need **embeddings** — dense vectors where similar words are close.
- **Next:** RNNs → embeddings → attention → Transformers — and the same **"reuse, don't reinvent"** transfer-learning mindset, now for language.

**Next week:** we start building sequence models, following Andrew Ng's Sequence Models course.